<ul>

# 0. Imports

In [45]:
import cda2
import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
import json
import re

from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# 1. Connect to Spark

In [46]:
api = cda2.Api()

Set configuration parameters to better optimize queries.

In [47]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use. 400 is quite high, but we'll be running 1 year at a time and want to have it done in just a few minutes.

In [48]:
api.start_spark(n_executors=400, config=config)

# 2. Define global variables and functions

In [49]:
year0 = "2019"
year1 = str(int(year0) + 1)

In [50]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 +"-01-01"}

In [51]:
def get_alpha_portion(route_name:T.StringType) -> str:
    regex = "^([a-zA-Z]+)\d+"
    match = re.search(regex, route_name)
    if match:
        output = match.group(1)
    else:
        output = ""
    return output

get_alpha_portion_udf = F.udf(get_alpha_portion, T.StringType())

result = get_alpha_portion("AB100")
print(result)

AB


In [52]:
def get_numeric_portion(route_name:T.StringType) -> int:
    regex = "^[a-zA-Z]+(\d+)"
    match = re.search(regex, route_name)
    if match:
        output = int(match.group(1))
    else:
        output = null
    return output

get_numeric_portion_udf = F.udf(get_numeric_portion, T.StringType())

#result = get_numeric_portion("AB100")
#print(result)

In [53]:
def pad_seq_with_zeros(seq:T.IntegerType) -> str:
    output = str(seq).zfill(5)
    return output

pad_seq_with_zeros_udf = F.udf(pad_seq_with_zeros, T.StringType())

#result = pad_seq_with_zeros("170")
#print(result)

# 3. Retrieve all routes

<li>only get SID and STAR procedure types. (procedure_type = "APPROACH" is ignored)</li>
<li>create a unique <i>grouping</i> column.</li>
</br>
this will retrieve multiple versions of each fix, one for each update cycle in the year.

In [55]:
df_routes_raw = (
    api.dataframe("ArincAirway", **dates, metadata=True)
    .select(
        F.col("identifier").alias("route"),
        F.col("arinc_record_info.customer_area_code").alias("area_code"),
#        "legs",
        F.explode("legs").alias("leg"),
        F.col("leg.path_terminator.identification.icao_region").alias("icao_region"),
        F.col("leg.path_terminator.identification.name").alias("fix_name"),
        F.col("leg.sequence_number").alias("seq"),
        F.col("metadata.effective_end_date").alias("end_date"),
        "path_terminators",
    )
    .withColumn("grouping", F.concat("route", F.lit("_"), "icao_region", F.lit("_"), pad_seq_with_zeros_udf("seq")))
    .withColumn("route_uniq", F.concat("route", F.lit("("), "area_code", F.lit("/"), "icao_region", F.lit(")")))
    .withColumn("fix_name_uniq", F.concat("fix_name", F.lit("("), "icao_region", F.lit(")")))
    .withColumn("alpha", get_alpha_portion_udf("route"))
    .withColumn("numeric", get_numeric_portion_udf("route").cast("int"))
    .filter(F.col("route") == "V35")
    .filter(F.col("area_code") == "USA")
#    .filter(F.col("icao_region") == "K6")
    .drop("legs", "leg")
#    .orderBy("grouping")
#    .orderBy("alpha", "numeric", "area_code", "icao_region", "seq")
)

#df_routes_raw.show()
#df_routes_raw.printSchema()

## > keep only the newest update

In [57]:
window = Window.partitionBy("grouping").orderBy(col("end_date").desc())
#window = Window.partitionBy("route").orderBy(col("end_date").desc())

df_routes_newest = (df_routes_raw
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row")
#    .orderBy("alpha", "numeric", "area_code", "icao_region", "seq")
#    .orderBy("fix_name")
)

df_routes_newest.show(200)

+-----+---------+-----------+--------+----+-------------+--------------------+------------+-----------+-------------+-----+-------+
|route|area_code|icao_region|fix_name| seq|     end_date|    path_terminators|    grouping| route_uniq|fix_name_uniq|alpha|numeric|
+-----+---------+-----------+--------+----+-------------+--------------------+------------+-----------+-------------+-----+-------+
|  V35|      USA|         K6|     GZG| 480|1577923200000|DHP:CURVE:WORPP:D...|V35_K6_00480|V35(USA/K6)|      GZG(K6)|    V|     35|
|  V35|      USA|         K6|   ZOMAD| 485|1577923200000|DHP:CURVE:WORPP:D...|V35_K6_00485|V35(USA/K6)|    ZOMAD(K6)|    V|     35|
|  V35|      USA|         K6|   STACY| 490|1577923200000|DHP:CURVE:WORPP:D...|V35_K6_00490|V35(USA/K6)|    STACY(K6)|    V|     35|
|  V35|      USA|         K6|   SLINK| 500|1577923200000|DHP:CURVE:WORPP:D...|V35_K6_00500|V35(USA/K6)|    SLINK(K6)|    V|     35|
|  V35|      USA|         K6|   MACET| 510|1577923200000|DHP:CURVE:WORPP:D..

In [ ]:
#df_routes_newest.show(truncate=False)

## > output all routes

In [12]:
df_routes_output = (
    df_routes_newest
        .withColumn("id", F.monotonically_increasing_id())
        .groupBy("route_uniq", "route", "alpha", "numeric", "area_code", "icao_region")
        .agg(F.concat_ws(":", F.collect_list("fix_name")).alias("fix_names"), F.concat_ws(":", F.collect_list("fix_name_uniq")).alias("uniq_fix_names"))
        .orderBy("alpha", "numeric", "area_code", "icao_region")
)

df_routes_output.write.option("header", True).csv("CRAFT/" + year0 + "/routes/all_routes", compression="None", mode="overwrite")

#df_routes_output.show()

# ------------------------------------

# 4. retrieve fixes

In [14]:
df_fixes_raw = (
    api.dataframe("ArincFix", **dates, metadata=True)
    .withColumn("fix_name_uniq", F.concat("identification.name", F.lit("("), "identification.icao_region", F.lit(")")))
    .select(
        "fix_name_uniq",
        F.col("identification.name").alias("fix_name"),
        F.col("arinc_record_info.customer_area_code").alias("area_code"),
        F.col("identification.icao_region").alias("icao_region"),
        "latitude",
        "longitude",
        F.col("magnetic_variation.modeled").alias("magnetic_variation"),
        F.col("metadata.effective_end_date").alias("end_date"),
    )
)

#df_fixes_raw.show()

Multiple versions found: 3.1.12.2, 3.1.22


In [16]:
df_fixes_raw.show()

+-------------+--------+---------+-----------+------------------+-------------------+------------------+-------------+
|fix_name_uniq|fix_name|area_code|icao_region|          latitude|          longitude|magnetic_variation|     end_date|
+-------------+--------+---------+-----------+------------------+-------------------+------------------+-------------+
|    00HNL(PH)|   00HNL|      PAC|         PH|19.935016666666666| -158.9449972222222|           9.25658|1573084800000|
|    00LIH(PH)|   00LIH|      PAC|         PH|23.348605555555555|-160.35078333333334|          9.432852|1573084800000|
|    00MKK(PH)|   00MKK|      PAC|         PH|22.537972222222223|-156.18695833333334|          9.644499|1573084800000|
|    23LIH(PH)|   23LIH|      PAC|         PH| 23.66581111111111|-160.58662777777778|           9.43936|1573084800000|
|    50MKK(PH)|   50MKK|      PAC|         PH|23.235561111111114| -155.6893638888889|          9.754545|1573084800000|
|       AA(K7)|      AA|      USA|         K7| 3

## > keep only the newest update

In [17]:
window = Window.partitionBy("fix_name_uniq").orderBy(col("end_date").desc())

df_fixes_newest = (df_fixes_raw
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row", "end_date")
    .orderBy("fix_name_uniq")
)

#df_fixes_newest.show()

In [18]:
df_routes_stripped = (
    df_routes_newest
    .select(
        "route",
        "route_uniq",
        "fix_name_uniq",
    )
)

#df_routes_stripped.show()

In [19]:
df_route_fixes = (
    df_fixes_newest
    .join(df_routes_stripped, on=["fix_name_uniq"], how="inner")
    .orderBy("fix_name_uniq", "route_uniq")
)

#df_route_fixes.show()

In [20]:
df_route_fixes_output = (
    df_route_fixes
        .groupBy("fix_name_uniq", "fix_name", "area_code", "icao_region", "latitude", "longitude", "magnetic_variation")
        .agg(F.concat_ws(":", F.collect_list("route")).alias("routes"), F.concat_ws(":", F.collect_list("route_uniq")).alias("routes_uniq_using_fix"))
        .orderBy("fix_name_uniq")
)

df_route_fixes_output.write.option("header", True).csv("CRAFT/" + year0 + "/routes/all_route_fixes", compression="None", mode="overwrite")

#df_route_fixes_output.show()